Task_1 : Setup & EDA

Execution Strategy

The Regex Layer: The clean_text_normalization logic addresses the engineering insight directly. It uses target boundary patterns (\bx{2,}\b) to eliminate structural anonymization tokens (XXXX) seamlessly before calculating the true word count.

Log-Scale Plotting: Consumer complaints often feature a heavy-tailed power law distribution (some complaints are essays). Plotting with log_scale=True ensures your histogram handles the skew visually without muddying data transparency.

Memory Safety: It leverages vector-friendly Pandas extensions and performs in-place assignment logic wherever possible to control memory strain when loading larger segments of raw CFPB sets.

In [2]:
import os
import re
import logging
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Align look-and-feel variables across environments
logging.basicConfig(level=logging.INFO)
sns.set_theme(style="whitegrid")

# Comprehensive taxonomy alignment dictionary covering all 4 target verticals
TARGET_PRODUCT_MAPPING = {
    'Credit card or prepaid card': 'Credit Card',
    'Credit card': 'Credit Card',
    'Consumer Loan': 'Personal Loan',
    'Student loan': 'Personal Loan',
    'Checking or savings account': 'Savings Account',
    'Savings account': 'Savings Account',
    'Money transfer, virtual currency, or money service': 'Money Transfer'
}

def load_and_filter_data(filepath: str) -> pd.DataFrame:
    logging.info(f"Reading target records incrementally from: {filepath}")
    try:
        # Optimized processing boundary chunksize
        chunks = pd.read_csv(filepath, low_memory=False, chunksize=500000)
    except Exception as e:
        logging.error(f"Error accessing file stream: {e}")
        return pd.DataFrame()
        
    filtered_chunks = []
    
    for chunk in chunks:
        # Isolate exactly matching data components
        chunk_filtered = chunk[chunk["Product"].isin(TARGET_PRODUCT_MAPPING.keys())].copy()
        chunk_filtered["Standardized_Product"] = chunk_filtered["Product"].map(TARGET_PRODUCT_MAPPING)
        filtered_chunks.append(chunk_filtered)
        
    if filtered_chunks:
        df_final = pd.concat(filtered_chunks, ignore_index=True)
    else:
        df_final = pd.DataFrame()
        
    return df_final

def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    print("\n--- Step 2: Handling Missing Values ---")
    before_count = len(df)

    df["Consumer complaint narrative"] = df["Consumer complaint narrative"].astype(str).str.strip()
    df.loc[df["Consumer complaint narrative"] == "", "Consumer complaint narrative"] = None
    df.loc[df["Consumer complaint narrative"] == "nan", "Consumer complaint narrative"] = None

    df = df.dropna(subset=["Consumer complaint narrative"]).copy()
    print(f"Records after clearing null/missing items: {len(df):,}")
    return df

def clean_text_normalization(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()

    boilerplate_patterns = [
        r"i am writing to file a complaint[^.]*\.?",
        r"to whom it may concern[^.]*\.?",
    ]
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, "", text)

    text = re.sub(r"\bx{2,}\b", "", text)
    text = re.sub(r"x{2,}([-/ ]x{2,})+", "", text)
    text = re.sub(r"x{2,}", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def process_text_and_filter_outliers(df: pd.DataFrame) -> pd.DataFrame:
    print("\n--- Step 3 & 4: Text Cleaning & Statistical Filtering ---")
    df["Cleaned_Narrative"] = df["Consumer complaint narrative"].apply(clean_text_normalization)
    df["Word_Count"] = df["Cleaned_Narrative"].apply(lambda x: len(x.split()))

    lower_bound = 10
    upper_bound_flag = int(df["Word_Count"].quantile(0.95))

    df_clean = df[df["Word_Count"] >= lower_bound].copy()

    # Layout visualization
    plt.figure(figsize=(10, 6))
    sns.histplot(df_clean["Word_Count"], bins=50, kde=True, color="teal", log_scale=True)
    plt.axvline(x=lower_bound, color="red", linestyle="--", label=f"Lower Filter Bound ({lower_bound} words)")
    plt.axvline(x=upper_bound_flag, color="orange", linestyle="--", label=f"Chunking Threshold ({upper_bound_flag} words)")
    plt.title("Distribution of CFPB Complaint Narrative Word Counts (Log Scale)")
    plt.xlabel("Word Count (Log Scale)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig("cfpb_narrative_word_count_distribution.png", dpi=300)
    plt.close()
    
    return df_clean

# --- End-to-End Notebook Validation Execution ---
if __name__ == "__main__":
    raw_input_path = "../data/raw/complaints.csv"
    canonical_output_path = "../data/processed/filtered_complaints.csv"
    
    # For standalone testing fallback, build comprehensive validation mock data
    if not os.path.exists(raw_input_path):
        raw_input_path = "mock_complaints.csv"
        mock_data = {
            "Product": [
                "Credit card or prepaid card", 
                "Student loan", 
                "Consumer Loan", 
                "Checking or savings account", 
                "Savings account",
                "Unrelated Categorical Entry"
            ],
            "Consumer complaint narrative": [
                "Fraudulent transaction on my card xxxx-xxxx.",
                "Student loan accumulation overages collected.",
                "Personal loan baseline rate calculation problems.",
                "My account savings numbers look completely wrong.",
                "Savings transaction dispute.",
                "Filter this item out."
            ]
        }
        pd.DataFrame(mock_data).to_csv(raw_input_path, index=False)

    # Run the ingestion pipelines
    df_raw = load_and_filter_data(raw_input_path)
    df_valid_text = handle_missing_values(df_raw)
    final_processed_df = process_text_and_filter_outliers(df_valid_text)
    
    # CRITICAL: Save the Canonical Dataset Artifact explicitly for downstream tasks
    output_directory = os.path.dirname(canonical_output_path)
    if output_directory and not os.path.exists(output_directory):
        os.makedirs(output_directory, exist_ok=True)
        
    final_processed_df.to_csv(canonical_output_path, index=False)
    print(f"\n[SUCCESS] Canonical pipeline artifact saved with {len(final_processed_df)} rows at: '{canonical_output_path}'")

INFO:root:Reading target records incrementally from: ../data/raw/complaints.csv



--- Step 2: Handling Missing Values ---
Records after clearing null/missing items: 489,511

--- Step 3 & 4: Text Cleaning & Statistical Filtering ---

[SUCCESS] Canonical pipeline artifact saved with 487053 rows at: '../data/processed/filtered_complaints.csv'
